# Lab 5: Table Manipulation and Visualization


**Helpful Resource:**
- [Python Reference](http://data8.org/sp24/reference/): Cheat sheet of helpful array & table methods!

**Recommended Reading**:
* [Visualization](https://inferentialthinking.com/chapters/07/Visualization.html)

In [ ]:
!apt-get install texlive texlive-xetex texlive-latex-extra pandoc
!pip install pypandoc
!pip install datascience

In [ ]:
# Connect Google Drive to Colab so you can access your files
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/Colab Notebooks/')

In [ ]:
# Don't change this cell; just run it.
import numpy as np
from datascience import *
import warnings
warnings.simplefilter('ignore', FutureWarning)

# These lines do some fancy plotting magic.\n",
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

## 1. Unemployment

The Great Recession of 2008-2009 was a period of economic decline observed globally, with scale and timing varying from country to country. In the United States, it resulted in a rapid rise in unemployment that affected industries and population groups to different extents.

The Federal Reserve Bank of St. Louis publishes data about jobs in the US.  Below, we've loaded data on unemployment in the United States. There are many ways of defining unemployment, and our dataset includes two notions of the unemployment rate:

1. *Non-Employment Index (or NEI)*: Among people who are able to work and are looking for a full-time job, the percentage who can't find a job.
2. *NEI-PTER*: Among people who are able to work and are looking for a full-time job, the percentage who can't find any job *or* are only working at a part-time job.  The latter group is called "Part-Time for Economic Reasons", so the acronym for this index is NEI-PTER.  (Economists are great at marketing.)

The source of the data is [here](https://fred.stlouisfed.org/categories/33509).

**Question 1.** The data are in a CSV file called `unemployment.csv`.  Load that file into a table called `unemployment`.

_Hint:_ After loading in the CSV file, the `unemployment` table should look like this:

<img src="https://github.com/data-8/materials-sp24/blob/main/hw/hw03/unemployment.png?raw=1" width="20%"/>


In [ ]:
unemployment = Table.read_table('./DS/unemployment.csv')
unemployment

**Question 2.** Sort the data in descending order by NEI, naming the sorted table `by_nei`.  Create another table called `by_nei_pter` that's sorted in descending order by NEI-PTER instead.


In [ ]:
by_nei = unemployment.sort('NEI', descending=True)
by_nei_pter = unemployment.sort('NEI-PTER', descending=True)

In [ ]:
# Run this cell to check your by_nei table. You do not need to change the code.
by_nei.show(5)

In [ ]:
# Run this cell to check your by_nei_pter table. You do not need to change the code.
by_nei_pter.show(5)

**Question 3.** Using `take`, assign `greatest_nei` to a table containing the data for the 11 quarters when NEI was greatest.

`greatest_nei` should be sorted in descending order of `NEI`. Note that each row of `unemployment` represents a quarter.


In [ ]:
greatest_nei = by_nei.take(np.arange(11))
greatest_nei

**Question 4.** It's believed that many people became PTER (recall: "Part-Time for Economic Reasons") in the "Great Recession" of 2008-2009.  NEI-PTER is the percentage of people who are unemployed (included in the NEI) plus the percentage of people who are PTER.

Compute an array containing the percentage of people who were PTER in each quarter.  (The first element of the array should correspond to the first row of `unemployment`, and so on.)

*Note:* Use the original `unemployment` table for this.


In [ ]:
pter = unemployment.column('NEI-PTER') - unemployment.column('NEI')
pter

**Question 5.** Add `pter` as a column to `unemployment` (name the column `PTER`) and sort the resulting table by that column in descending order.  Call the resulting table `by_pter`.

Try to do this with a single line of code, if you can.


In [ ]:
by_pter = unemployment.with_columns('PTER', pter).sort('PTER', descending=True)
by_pter

**Question 6.** Create a line plot of PTER over time. To do this, create a new table called `pter_over_time` with the same columns as the `unemployment` table with the addition of two new columns: `Year` and `PTER` using the `year` array and the `pter` array, respectively. Then, generate a line plot using one of the table methods you've learned in class.

The order of the columns matter for our correctness tests, so be sure `Year` comes before `PTER`.

*Note:* When constructing `pter_over_time`, do not just add the `year` column to the `by_pter` table. Please follow the directions in the question above.


In [ ]:
year = 1994 + np.arange(unemployment.num_rows)/4
pter_over_time = unemployment.with_columns('Year', year, 'PTER', pter)
pter_over_time.plot('Year', 'PTER')
plt.ylim(0,2); # Do not change this line

**Question 7.** Were PTER rates high during the Great Recession (that is to say, were PTER rates particularly high in the years 2008 through 2011)? Assign `highPTER` to `True` if you think PTER rates were high in this period, or `False` if you think they weren't.


In [ ]:
highPTER = True

## 2. Birth Rates

The following table gives Census-based population estimates for each US state on both July 1, 2015 and July 1, 2016. The last four columns describe the components of the estimated change in population during this time interval. **For all questions below, assume that the word "states" refers to all 52 rows including Puerto Rico and the District of Columbia.**

The data was taken from [here](http://www2.census.gov/programs-surveys/popest/datasets/2010-2016/national/totals/nst-est2016-alldata.csv). (Note: If the file doesn't download for you when you click the link, you can copy and paste the link address it into your address bar!) If you want to read more about the different column descriptions, click [here](http://www2.census.gov/programs-surveys/popest/datasets/2010-2015/national/totals/nst-est2015-alldata.pdf).

The raw data is a bit messy—run the cell below to clean the table and make it easier to work with.

In [ ]:
# Don't change this cell; just run it.
pop = Table.read_table('./DS/nst-est2016-alldata.csv').where('SUMLEV', 40).select([1, 4, 12, 13, 27, 34, 62, 69])
pop = pop.relabeled('POPESTIMATE2015', '2015').relabeled('POPESTIMATE2016', '2016')
pop = pop.relabeled('BIRTHS2016', 'BIRTHS').relabeled('DEATHS2016', 'DEATHS')
pop = pop.relabeled('NETMIG2016', 'MIGRATION').relabeled('RESIDUAL2016', 'OTHER')
pop = pop.with_columns("REGION", np.array([int(region) if region != "X" else 0 for region in pop.column("REGION")]))
pop.set_format([2, 3, 4, 5, 6, 7], NumberFormatter(decimals=0)).show(5)

**Question 1.** Assign `us_birth_rate` to the total US annual birth rate during this time interval. The annual birth rate for a year-long period is the total number of births in that period as a proportion of the total population size at the start of the time period.

_Hint:_ Remember that each row in the `pop` table refers to a state, not the US as a whole.


In [ ]:
us_birth_rate = sum(pop.column('BIRTHS')) / sum(pop.column('2015'))
us_birth_rate

**Question 2.** Assign `movers` to the number of states for which the **absolute value** of the **annual rate of migration** was higher than 1%. The annual rate of migration for a year-long period is the net number of migrations (in and out) as a proportion of the population size at the start of the period. The `MIGRATION` column contains estimated annual net migration counts by state.

*Hint*: `migration_rates` should be a table and `movers` should be a number.


In [ ]:
migration_rates = pop.column('MIGRATION') / pop.column('2015')
movers = sum(abs(migration_rates) > 0.01)
movers

**Question 3.** Assign `west_births` to the total number of births that occurred in region 4 (the Western US).

*Hint:* Make sure you double check the type of the values in the `REGION` column and appropriately filter (i.e. the types must match!).


In [ ]:
west_births = sum(pop.where('REGION', 4).column('BIRTHS'))
west_births

**Question 4.** In the next question, you will be creating a visualization to understand the relationship between birth and death rates. The annual death rate for a year-long period is the total number of deaths in that period as a proportion of the population size at the start of the time period.

What visualization is most appropriate to see if there is an association between annual birth and death rates across multiple states in the United States?

1. Line Graph
2. Bar Chart
3. Scatter Plot

Assign `visualization` below to the number corresponding to the correct visualization.


In [ ]:
visualization = 3

<!-- BEGIN QUESTION -->

**Question 5.** In the code cell below, create a visualization that will help us determine if there is an association between birth rate and death rate during this time interval. It may be helpful to create an intermediate table containing the birth and death rates for each state.

Things to consider:

- What type of chart will help us illustrate an association between 2 variables?
- How can you manipulate a certain table to help generate your chart?

In [ ]:
birth_rates_2015 = pop.column('BIRTHS') / pop.column('2015')
death_rates_2015 = pop.column('DEATHS') / pop.column('2015')
pop_with_rates = pop.with_columns(
    'Birth Rate', birth_rates_2015,
    'Death Rate', death_rates_2015
)
pop_with_rates.scatter('Birth Rate', 'Death Rate')

<!-- END QUESTION -->

**Question 6.** True or False: There is an association between birth rate and death rate during this time interval.

Assign `assoc` to `True` or `False` in the cell below.


In [ ]:
assoc = True

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a pdf file for you to submit. **Please save before exporting!**

In [ ]:
# should change the directory and file name matching to yours
!jupyter nbconvert './DS/lab05(SSU).ipynb' --to pdf